# FG-TRACER: fgtracer-gemma

Traces **one** model only (so a single Kaggle session never approaches the 12h
cap). Work is split across the two T4 GPUs and merged. Outputs land in
`/kaggle/working/outputs_gemma` and are saved as this kernel's output.

**One-time setup:** attach as inputs (1) the model and (2) the `fgtracer-code`
dataset (the driver `run_kaggle.sh` uploads the code for you).

In [ ]:
# 1. Deps. Single clean pip pass (no destructive uninstall, no --no-deps numpy
#    hack -- those left transformers half-installed so GenerationMixin failed to
#    import). Pin transformers to the 4.56 line (requirements: >=4.50,<5) and let
#    pip resolve the rest in ONE pass. Fail loudly on the exact imports we need.
%pip install -q -U "transformers>=4.56,<4.57" "huggingface_hub>=0.34" accelerate "datasets>=2.19" pillow safetensors
import transformers, torch
from transformers import AutoModelForImageTextToText
from transformers.generation import GenerationMixin
print('deps OK | transformers', transformers.__version__,
      '| torch', torch.__version__, '| GPUs', torch.cuda.device_count())


In [ ]:
# extract/copy FG-TRACER code into the working dir (handles zip OR loose files)
import glob, os, shutil, zipfile
WORK='/kaggle/working'
zips=glob.glob('/kaggle/input/**/fgtracer_code.zip',recursive=True)
if zips:
    zipfile.ZipFile(zips[0]).extractall(WORK)
else:
    src=glob.glob('/kaggle/input/**/run_tracing.py',recursive=True)
    assert src,'attach the fgtracer-code dataset'
    cd=os.path.dirname(src[0])
    for it in os.listdir(cd):
        s=os.path.join(cd,it); d=os.path.join(WORK,it)
        if os.path.isdir(s): shutil.copytree(s,d,dirs_exist_ok=True)
        elif not os.path.exists(d): shutil.copy2(s,d)
os.chdir(WORK)
import fgtracer  # fail loudly if the package did not land
print('code ready:',sorted(os.listdir(WORK))[:12])

In [ ]:
# 3. Resolve the model checkpoint dir (from attached model input)
import glob, os
MATCH, NOT = 'gemma-3', 'medgemma'
cand = []
for cfg in glob.glob('/kaggle/input/**/config.json', recursive=True):
    d = os.path.dirname(cfg); low = d.lower()
    if MATCH in low and NOT not in low:
        cand.append(d)
assert cand, f'model dir matching {MATCH!r} not found under /kaggle/input'
CKPT = sorted(cand, key=len)[0]
print('checkpoint:', CKPT)

In [ ]:
# 4. Trace, split across the two T4s (disjoint sample ranges), then merge.
#    Each GPU writes its OWN dir (no file races); ids are globally unique.
import subprocess, os
OUT = 'outputs_gemma'
common = ['--checkpoint_dir', CKPT, '--device', 'cuda', '--dtype', 'bfloat16',
          '--quantization', 'none', '--block_types', 'image_to_last,question_to_last',
          '--k', '1', '--max_new_tokens', '64']
jobs = [
    (dict(os.environ, CUDA_VISIBLE_DEVICES='0'),
     ['--start_index', '0',      '--num_samples', '50', '--answer_path', OUT + '_g0']),
    (dict(os.environ, CUDA_VISIBLE_DEVICES='1'),
     ['--start_index', '50', '--num_samples', '50', '--answer_path', OUT + '_g1']),
]
procs = []
for env, extra in jobs:
    if int(env['CUDA_VISIBLE_DEVICES']) >= __import__('torch').cuda.device_count():
        continue  # only one GPU available -> skip the second job
    procs.append(subprocess.Popen(['python', 'run_tracing.py'] + common + extra, env=env))
for p in procs:
    p.wait()
# merge whichever part-dirs exist
parts = [d for d in (OUT + '_g0', OUT + '_g1') if os.path.isdir(d)]
subprocess.run(['python', 'merge_outputs.py', '--inputs', *parts, '--output', OUT], check=True)
print('merged ->', OUT, sorted(os.listdir(OUT)))

In [ ]:
# 5. Quick sanity on the fresh results (real answers, not 8-token stubs)
import json, glob
for f in sorted(glob.glob('outputs_gemma/results_*_k1.json')):
    d = json.load(open(f)); rs = d['results']
    print(os.path.basename(f), '| n=', len(rs),
          '| max_new_tokens=', d['meta']['max_new_tokens'],
          '| acc=', round(d['run_summary'].get('accuracy') or 0, 3))
    print('   sample gen:', repr(rs[0]['generated'][:70]))